In [2]:
!pip install -q nltk textblob spacy spacytextblob
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 12.0 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import spacy
from spacy.tokens import Doc
# Customer reviews
reviews = [
"The doctor was very caring and helpful.",
"The hospital service was terrible.",
"The waiting time was too long.",
"The hospital staff is excellent.",
"The service was okay, nothing special."
]
# =====================================================
# 1. VADER
# =====================================================
vader = SentimentIntensityAnalyzer()
print("========== VADER ==========")
for review in reviews:
  scores = vader.polarity_scores(review)
  compound = scores["compound"]
  if compound >= 0.05:
    sentiment = "Positive"
  elif compound <= -0.05:
    sentiment = "Negative"
  else:
    sentiment = "Neutral"
    print("\nReview:", review)
    print("Polarity (Compound):", compound)
    print("Positive:", scores["pos"])
    print("Negative:", scores["neg"])
    print("Neutral :", scores["neu"])
    print("Sentiment:", sentiment)


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


========== VADER ==========

Review: The waiting time was too long.
Polarity (Compound): 0.0
Positive: 0.0
Negative: 0.0
Neutral : 1.0
Sentiment: Neutral


In [7]:
print("\n========== TEXTBLOB ==========")
for review in reviews:
  blob = TextBlob(review)
  polarity = blob.sentiment.polarity
  subjectivity = blob.sentiment.subjectivity
  if polarity > 0:
    sentiment = "Positive"
  elif polarity < 0:
    sentiment = "Negative"
  else:
    sentiment = "Neutral"
  print("\nReview:", review)
  print("Polarity:", polarity)
  print("Subjectivity:", subjectivity)
  print("Sentiment:", sentiment)




========== TEXTBLOB ==========

Review: The doctor was very caring and helpful.
Polarity: 0.2
Subjectivity: 0.3
Sentiment: Positive

Review: The hospital service was terrible.
Polarity: -1.0
Subjectivity: 1.0
Sentiment: Negative

Review: The waiting time was too long.
Polarity: -0.05
Subjectivity: 0.4
Sentiment: Negative

Review: The hospital staff is excellent.
Polarity: 1.0
Subjectivity: 1.0
Sentiment: Positive

Review: The service was okay, nothing special.
Polarity: 0.4285714285714286
Subjectivity: 0.5357142857142857
Sentiment: Positive


In [41]:
# 1. Lexical diversity
for i, text in enumerate(articles):
    words = text.lower().split()
    diversity = len(set(words)) / len(words)
    print("Article", i+1, "Lexical Diversity:", round(diversity, 2))

Article 1 Lexical Diversity: 0.9
Article 2 Lexical Diversity: 0.89
Article 3 Lexical Diversity: 1.0
Article 4 Lexical Diversity: 1.0


In [8]:
# =====================================================
# 3. spaCy + TextBlob (Custom Component)
# =====================================================
print("\n========== SPACY + TEXTBLOB (Custom Component) ==========")
# Load spaCy English model
nlp = spacy.load("en_core_web_sm")
# Define custom extensions for polarity and subjectivity
if not Doc.has_extension("polarity"):
  Doc.set_extension("polarity", default=0.0)
if not Doc.has_extension("subjectivity"):
  Doc.set_extension("subjectivity", default=0.0)
# Create a custom spaCy pipeline component using TextBlob's sentiment
@spacy.language.Language.component("textblob_sentiment_component")
def textblob_sentiment_component(doc):
  blob = TextBlob(doc.text)
  doc._.polarity = blob.sentiment.polarity
  doc._.subjectivity = blob.sentiment.subjectivity
  return doc
# Add the custom component to the spaCy pipeline
nlp.add_pipe("textblob_sentiment_component", last=True)



========== SPACY + TEXTBLOB (Custom Component) ==========


<function __main__.textblob_sentiment_component(doc)>